# Bangladesh: 0115_xee_mann_kendall

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0115_xee_mann_kendall
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'methods' / '0115_xee_mann_kendall'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""GEE_0023_mannkendall_xee.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-120: Spatio-Temporal Trend Analysis, using Mann-Kendall Test in Python API (Xee)

This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
import geemap
import xarray as xr
import xee
None
None
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
collection = ee.ImageCollection('ECMWF/ERA5/MONTHLY').select('mean_2m_air_temperature').filterDate('1980', '2025')
collection
ds = xr.open_dataset(collection, engine='ee', crs='EPSG:4326', scale=0.27, geometry=BD_ROI)
ds
import pymannkendall as mk
help(mk.original_test)

def mk_test(img):
    test = mk.original_test(img)
    trend = test.trend
    trend_class = {'increasing': 1, 'decreasing': -1, 'no_trend': 0}
    trend_reclass = trend_class.get(trend, 0)
    score = test.s
    pval = test.p
    return (trend_reclass, score, pval)
help(xr.apply_ufunc)
change_trend, change_score, change_pval = xr.apply_ufunc(mk_test, ds.mean_2m_air_temperature, input_core_dims=[['time']], output_core_dims=[[], [], []], dask='allowed', vectorize=True, output_dtypes=['int', 'float64', 'float64'])
change_trend = change_trend.rename('trend')
change_score = change_score.rename('score') * 0.0001
change_pval = change_pval.rename('p_value')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(8, 2.3))
plt.tight_layout()
change_trend.plot(x='lon', y='lat', ax=ax[0])
change_score.plot(x='lon', y='lat', ax=ax[1], robust=True)
change_pval.plot(x='lon', y='lat', ax=ax[2], robust=True, vmax=0.05)
for axis in ax:
    axis.set_ylabel('')
    axis.set_xlabel('')
    axis.set_xticks([])
    axis.set_yticks([])
plt.savefig('mk_change.png', dpi=360, bbox_inches='tight')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'methods', '0115_xee_mann_kendall', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'methods')  # explicit upload; requires Drive OAuth scope
